<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/thermodynamics/readproperties.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reading thermodynamic and physical properties with NeqSim

This notebook develops a reproducible workflow for creating a rich natural-gas fluid, performing a two-phase TP flash, and extracting properties needed in process and equipment calculations.

## Learning objectives

After completing the notebook, you can:

1. install and identify the public PyPI release of NeqSim used in a calculation;
2. distinguish equilibrium (thermodynamic) properties from transport (physical) properties;
3. read phase fractions, composition, density, heat capacity, viscosity, and thermal conductivity with explicit units;
4. organize results in engineering tables and verify basic closure and plausibility;
5. investigate pressure trends without accidentally reusing stale phase properties.

## Engineering context and theory

A **TP flash** determines the stable phase split at specified temperature $T$, pressure $P$, and overall composition $z_i$. For each component, material closure requires

$$z_i = \sum_{\alpha=1}^{N_p} \beta_\alpha x_{i,\alpha},$$

where $\beta_\alpha$ is the molar phase fraction and $x_{i,\alpha}$ is the component mole fraction in phase $\alpha$. The phase fractions satisfy $\sum_\alpha \beta_\alpha=1$.

Transport properties such as viscosity $\mu$ and thermal conductivity $k$ require physical-property initialization after the flash. Report every value with units and phase identity; a single unlabeled “density” is ambiguous in a multiphase system.

## 1. Colab-compatible setup

The setup installs the newest public PyPI package. Restarting the runtime is normally unnecessary because NeqSim starts the Java Virtual Machine when imported.

In [1]:
import subprocess, sys
import importlib.util
if importlib.util.find_spec("neqsim") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", "neqsim"], check=True)
else:
    print("neqsim is already installed; use the command below in a fresh Colab runtime if an upgrade is required.")
    print(f"{sys.executable} -m pip install --upgrade --no-cache-dir neqsim")

In [2]:
import importlib.metadata as metadata
import platform, subprocess

NEQSIM_VERSION = metadata.version("neqsim")
PYTHON_VERSION = platform.python_version()
JAVA_VERSION = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True).splitlines()[0]
print(f"NeqSim: {NEQSIM_VERSION}")
print(f"Python: {PYTHON_VERSION}")
print(f"Java:   {JAVA_VERSION}")

NeqSim: 3.16.0
Python: 3.13.5
Java:   openjdk version "21.0.10" 2026-01-20


## 2. Create and flash a representative rich gas

NeqSim’s built-in `rich gas` composition is public, compact, and reproducible. The SRK equation of state with the classic mixing rule is suitable for demonstrating hydrocarbon vapor–liquid equilibrium. The state below, 22.3 °C and 10 bara, intentionally gives gas and hydrocarbon-liquid phases.

In [3]:
from neqsim.thermo import createfluid, TPflash

fluid = createfluid("rich gas")
fluid.setMixingRule("classic")
fluid.setTemperature(22.3, "C")
fluid.setPressure(10.0, "bara")
TPflash(fluid)
fluid.initPhysicalProperties()

print("Number of phases:", fluid.getNumberOfPhases())
print("Phase types:", [str(fluid.getPhase(i).getPhaseTypeName()) for i in range(fluid.getNumberOfPhases())])

Number of phases: 2
Phase types: ['gas', 'oil']


`TPflash` establishes equilibrium. `initPhysicalProperties()` is then called explicitly so density, viscosity, and thermal conductivity correspond to the flashed state.

## 3. Extract phase properties with explicit units

The table separates equilibrium descriptors (phase fraction and composition) from caloric and transport properties. Heat capacities are reported on a mass basis.

In [4]:
import pandas as pd

rows = []
for i in range(fluid.getNumberOfPhases()):
    phase = fluid.getPhase(i)
    rows.append({
        "phase": str(phase.getPhaseTypeName()),
        "molar phase fraction": float(fluid.getBeta(i)),
        "molar mass [kg/mol]": float(phase.getMolarMass()),
        "density [kg/m3]": float(phase.getDensity("kg/m3")),
        "Cp [J/(kg K)]": float(phase.getCp("J/kgK")),
        "Cv [J/(kg K)]": float(phase.getCv("J/kgK")),
        "viscosity [Pa s]": float(phase.getViscosity("kg/msec")),
        "thermal conductivity [W/(m K)]": float(phase.getThermalConductivity("W/mK")),
        "Z [-]": float(phase.getZ()),
    })
properties = pd.DataFrame(rows).set_index("phase")
properties.round(6)

       molar phase fraction  ...     Z [-]
phase                        ...          
gas                0.939656  ...  0.968862
oil                0.060344  ...  0.067120

[2 rows x 8 columns]

### Interpretation

At this state the gas phase should have a low density and viscosity, while the hydrocarbon liquid should be much denser and more viscous. The liquid compressibility factor is not interpreted like an ideal-gas correction; it is mainly a model state variable for the dense phase.

## 4. Inspect phase compositions and component closure

In [5]:
components = [str(fluid.getComponent(i).getComponentName()) for i in range(fluid.getNumberOfComponents())]
composition = pd.DataFrame(index=components)
composition["overall z"] = [float(fluid.getComponent(i).getz()) for i in range(fluid.getNumberOfComponents())]
for p in range(fluid.getNumberOfPhases()):
    name = str(fluid.getPhase(p).getPhaseTypeName())
    composition[f"{name} x"] = [float(fluid.getPhase(p).getComponent(i).getx()) for i in range(fluid.getNumberOfComponents())]
composition.head(10).round(6)

           overall z     gas x     oil x
nitrogen    0.009074  0.009647  0.000156
CO2         0.018149  0.019140  0.002717
methane     0.744102  0.789341  0.039643
ethane      0.099819  0.104122  0.032800
propane     0.045372  0.045061  0.050218
i-butane    0.009074  0.008226  0.022280
n-butane    0.010889  0.009257  0.036302
i-pentane   0.009074  0.005914  0.058291
n-pentane   0.009074  0.005240  0.068786
n-hexane    0.009074  0.002573  0.110315

In [6]:
import numpy as np

betas = np.array([float(fluid.getBeta(i)) for i in range(fluid.getNumberOfPhases())])
x = np.array([[float(fluid.getPhase(p).getComponent(i).getx()) for i in range(fluid.getNumberOfComponents())] for p in range(fluid.getNumberOfPhases())])
z = composition["overall z"].to_numpy()
z_reconstructed = betas @ x
component_error = np.max(np.abs(z - z_reconstructed))
print(f"Sum of phase fractions: {betas.sum():.12f}")
print(f"Maximum component closure error: {component_error:.3e}")

Sum of phase fractions: 1.000000000000
Maximum component closure error: 2.220e-16


## 5. Pressure sweep

For each pressure, the same overall fluid is copied, flashed, and initialized again. This avoids reading properties left over from a previous state. The sweep is an engineering sensitivity study, not a phase-envelope calculation.

In [7]:
import matplotlib.pyplot as plt

records = []
for pressure in np.linspace(5.0, 50.0, 10):
    case = fluid.clone()
    case.setPressure(float(pressure), "bara")
    case.setTemperature(22.3, "C")
    TPflash(case)
    case.initPhysicalProperties()
    gas_index = next(i for i in range(case.getNumberOfPhases()) if str(case.getPhase(i).getPhaseTypeName()).lower() == "gas")
    records.append({
        "pressure [bara]": pressure,
        "gas fraction [-]": float(case.getBeta(gas_index)),
        "gas density [kg/m3]": float(case.getPhase(gas_index).getDensity("kg/m3")),
    })
pressure_results = pd.DataFrame(records)
pressure_results.round(5)

   pressure [bara]  gas fraction [-]  gas density [kg/m3]
0              5.0           0.95207              4.47421
1             10.0           0.93966              8.86853
2             15.0           0.93006             13.30823
3             20.0           0.92170             17.82253
4             25.0           0.91404             22.42814
5             30.0           0.90683             27.13718
6             35.0           0.89993             31.95961
7             40.0           0.89327             36.90428
8             45.0           0.88679             41.97935
9             50.0           0.88044             47.19250

In [8]:
ax = pressure_results.plot(x="pressure [bara]", y="gas density [kg/m3]", marker="o", legend=False)
ax.set_ylabel("Gas density [kg/m³]")
ax.set_title("Rich-gas density at 22.3 °C")
ax.grid(True)
plt.show()

The gas density is expected to rise with pressure over this range. The gas fraction can change because pressure shifts vapor–liquid equilibrium; therefore density trends should be interpreted together with the phase split.

## 6. Focused validation checks

These assertions catch common notebook failures: missing phases, uninitialized transport properties, unit mistakes, failed material closure, and nonphysical trend data.

In [9]:
assert fluid.getNumberOfPhases() == 2
assert set(properties.index.str.lower()) == {"gas", "oil"}
assert abs(betas.sum() - 1.0) < 1e-10
assert component_error < 1e-10
assert np.isfinite(properties.to_numpy()).all()
assert (properties["density [kg/m3]"] > 0).all()
assert properties.loc["oil", "density [kg/m3]"] > properties.loc["gas", "density [kg/m3]"]
assert properties.loc["oil", "viscosity [Pa s]"] > properties.loc["gas", "viscosity [Pa s]"]
assert pressure_results["gas density [kg/m3]"].is_monotonic_increasing
assert pressure_results[["gas fraction [-]", "gas density [kg/m3]"]].apply(np.isfinite).all().all()
assert pressure_results["gas fraction [-]"].between(0.0, 1.0).all()
print("All validation checks passed.")

All validation checks passed.


## Limitations and applicability

- Results depend on equation of state, mixing rule, component characterization, and transport-property correlations.
- A built-in synthetic rich gas is appropriate for teaching, not for certifying a specific field fluid.
- Close to critical conditions, phase identification and property trends require a dedicated phase-envelope and sensitivity study.
- For design work, compare key properties against laboratory PVT data or an accepted reference model and document uncertainty.

## References and next steps

- [NeqSim documentation](https://equinor.github.io/neqsimhome/)
- [NeqSim Python examples](https://github.com/equinor/neqsim-python)
- [NeqSim source repository](https://github.com/equinor/neqsim)

Exercises: repeat the sweep at a higher temperature; compare SRK and PR; replace the built-in gas with a measured composition; export the property table with units and model metadata.